# Chapter 7 · VQE for the H₂ Molecule

## Objectives

1. Understand how a quantum chemistry problem is mapped to a qubit Hamiltonian.
2. Implement the full VQE algorithm for H₂ (equilibrium geometry) using Qiskit Nature.
3. Calculate the dissociation energy curve comparing HF, exact FCI, and VQE-UCCSD.
4. Analyze the electronic correlation error captured by the UCCSD ansatz.

---

## 7A.1 The H₂ molecule as a quantum benchmark

The hydrogen molecule H₂ is the simplest system with electronic correlation. It consists of two electrons in two molecular orbitals (σ and σ*). In the STO-3G basis **4 qubits** are needed under the Jordan-Wigner mapping, and the UCCSD ansatz has only **3 variational parameters**.

The electronic Hamiltonian is expressed as Pauli operators via the Jordan-Wigner mapping:

$$\hat{H} = \sum_k h_k \hat{P}_k, \quad \hat{P}_k \in \{I, X, Y, Z\}^{\otimes n}$$

The expected value of the energy for a state $|\psi(\boldsymbol{\theta})\rangle$ produced by the UCCSD ansatz is:

$$E(\boldsymbol{\theta}) = \langle \psi(\boldsymbol{\theta}) | \hat{H} | \psi(\boldsymbol{\theta}) \rangle$$

and the VQE minimizes this quantity by optimizing the parameters $\boldsymbol{\theta}$.

---

## 7A.2 Workflow

```
PySCF driver  →  Fermionic Hamiltonian  →  JW mapping  →  Qubit Hamiltonian
                                                                   ↓
                                          UCCSD ansatz (HF + excitations)
                                                                   ↓
                                          VQE + SLSQP (classical optimization)
                                                                   ↓
                                          Total energy = E_elec + E_nucl
```

**Required dependencies:**
```bash
pip install qiskit qiskit-nature qiskit-algorithms pyscf
```

In [ ]:
import sys
from typing import List

import numpy as np
import matplotlib.pyplot as plt

from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD

from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import SLSQP
from qiskit.primitives import StatevectorEstimator

print("Environment:", sys.executable)
print("Modules loaded successfully.")

Environment: c:\Program Files\Python312\python.exe
Modules loaded successfully.


## 7A.3 Step 1 — Molecular specification

We define the H₂ geometry at the equilibrium bond length ($R = 0.7414$ Å) and the minimal STO-3G basis.

In [ ]:
# ── Equilibrium geometry ───────────────────────────────────────────
BOND_LENGTH = 0.7414      # Å (experimental H-H distance)
GEOMETRY    = f"H 0 0 0; H 0 0 {BOND_LENGTH}"
BASIS_SET   = "sto3g"     # Minimal basis: 2 spatial orbitals → 4 qubits (JW)
CHARGE      = 0
SPIN        = 0           # Singlet (multiplicity 1)

print(f"Molecule : H₂")
print(f"Geometry : {GEOMETRY}")
print(f"Basis    : {BASIS_SET}")
print(f"Charge   : {CHARGE}")
print(f"Spin     : {SPIN}  →  multiplicity {SPIN + 1}")

Molecule : H₂
Geometry : H 0 0 0; H 0 0 0.7414
Basis    : sto3g
Charge   : 0
Spin     : 0  →  multiplicity 1


## 7A.4 Step 2 — PySCF driver and fermionic Hamiltonian

The driver runs a classical Hartree-Fock (HF) calculation with PySCF to obtain the one-electron integrals $h_{pq}$ and two-electron integrals $h_{pqrs}$ that define the second-quantization electronic Hamiltonian:

$$\hat{H}_{el} = \sum_{pq} h_{pq} a_p^\dagger a_q + \frac{1}{2}\sum_{pqrs} h_{pqrs} a_p^\dagger a_q^\dagger a_s a_r$$

In [ ]:
# ── PySCF driver ───────────────────────────────────────────────────
driver = PySCFDriver(
    atom=GEOMETRY,
    basis=BASIS_SET,
    charge=CHARGE,
    spin=SPIN,
    unit=DistanceUnit.ANGSTROM,
)

driver_result = driver.run()
print("✓ PySCF driver executed.")

# Electronic system metadata
num_particles         = driver_result.num_particles
num_spatial_orbitals  = driver_result.num_spatial_orbitals
nuclear_repulsion     = float(driver_result.nuclear_repulsion_energy)
second_q_hamiltonian  = driver_result.hamiltonian.second_q_op()

print(f"\nParticles (α, β)     : {num_particles}")
print(f"Spatial orbitals     : {num_spatial_orbitals}")
print(f"Nuclear repulsion    : {nuclear_repulsion:.10f} Ha")

MissingOptionalLibraryError: "The 'pyscf' library is required to use 'PySCFDriver'.  See https://pyscf.org/install.html."


## 7A.5 Step 3 — Jordan-Wigner mapping

The Jordan-Wigner (JW) mapping converts fermionic creation/annihilation operators into Pauli strings. For $n$ spin orbitals, $n$ qubits are needed.

For the $k$-th mode the mapping is:

$$a_k^\dagger \to \left(\bigotimes_{j<k} Z_j\right) \otimes \frac{X_k - iY_k}{2}, \quad a_k \to \left(\bigotimes_{j<k} Z_j\right) \otimes \frac{X_k + iY_k}{2}$$

In [ ]:
# ── Jordan-Wigner mapping ──────────────────────────────────────────
mapper            = JordanWignerMapper()
qubit_hamiltonian = mapper.map(second_q_hamiltonian)

pauli_terms = qubit_hamiltonian.to_list()

print(f"Qubits required  : {qubit_hamiltonian.num_qubits}")
print(f"Pauli terms      : {len(pauli_terms)}")
print("\nFirst 8 terms of the Hamiltonian:")
for i, (pauli_str, coeff) in enumerate(pauli_terms[:8], start=1):
    print(f"  {i:2d}. {coeff:+.8f} · {pauli_str}")

## 7A.6 Step 4 — UCCSD ansatz

The **Unitary Coupled Cluster Singles and Doubles (UCCSD)** ansatz starts from the Hartree-Fock (HF) state and applies unitary excitations:

$$|\psi(\boldsymbol{\theta})\rangle = e^{\hat{T}(\boldsymbol{\theta}) - \hat{T}^\dagger(\boldsymbol{\theta})} |\psi_{\text{HF}}\rangle$$

where $\hat{T} = \hat{T}_1 + \hat{T}_2$ contains single and double excitations.

For H₂ in STO-3G there is only one relevant double excitation → **3 variational parameters**.

In [ ]:
# ── Hartree-Fock initial state ─────────────────────────────────────
hf_state = HartreeFock(
    num_spatial_orbitals=num_spatial_orbitals,
    num_particles=num_particles,
    qubit_mapper=mapper,
)

# ── UCCSD ansatz ───────────────────────────────────────────────────
ansatz = UCCSD(
    num_spatial_orbitals=num_spatial_orbitals,
    num_particles=num_particles,
    qubit_mapper=mapper,
    initial_state=hf_state,
)

print(f"Variational parameters : {ansatz.num_parameters}")
print(f"Circuit depth          : {ansatz.decompose().depth()}")
print()
print("UCCSD circuit (high level):")
print(ansatz.draw('text'))

## 7A.7 Step 5 — VQE execution

The VQE alternates between:
- **Quantum evaluation**: prepare $|\psi(\boldsymbol{\theta})\rangle$ and measure $\langle H \rangle$.
- **Classical optimization**: adjust $\boldsymbol{\theta}$ with SLSQP until $E(\boldsymbol{\theta})$ is minimized.

We use `StatevectorEstimator` (exact noiseless simulation).

In [ ]:
# ── VQE configuration ──────────────────────────────────────────────
estimator  = StatevectorEstimator()
optimizer  = SLSQP(maxiter=120, eps=1e-6)

energy_history: List[float] = []

def callback(eval_count, params, mean, metadata):
    """Records the energy at each optimizer evaluation."""
    energy_history.append(float(mean))
    if eval_count == 1 or eval_count % 5 == 0:
        print(f"  Eval {eval_count:3d}: E = {float(mean):+.10f} Ha")

vqe = VQE(
    estimator=estimator,
    ansatz=ansatz,
    optimizer=optimizer,
    callback=callback,
)

print("Running VQE...")
result = vqe.compute_minimum_eigenvalue(qubit_hamiltonian)
print("\n✓ VQE completed.")

In [ ]:
# ── Results ────────────────────────────────────────────────────────
vqe_electronic = float(np.real(result.eigenvalue))
vqe_total      = vqe_electronic + nuclear_repulsion

# H₂ STO-3G references at R = 0.7414 Å
HF_REF  = -1.11733   # Ha  (Hartree-Fock)
FCI_REF = -1.17463   # Ha  (Full CI, exact result in basis)

print("=" * 60)
print(f"VQE electronic energy     : {vqe_electronic:+.10f} Ha")
print(f"Nuclear repulsion         : {nuclear_repulsion:+.10f} Ha")
print(f"VQE total energy          : {vqe_total:+.10f} Ha")
print("-" * 60)
print(f"HF reference              : {HF_REF:+.5f} Ha")
print(f"FCI reference             : {FCI_REF:+.5f} Ha")
print(f"Error vs FCI              : {abs(vqe_total - FCI_REF) * 1000:.3f} mHa")
print(f"Captured correlation energy: {abs(vqe_total - HF_REF) / abs(FCI_REF - HF_REF) * 100:.1f}%")
print("=" * 60)
print(f"\nOptimal parameters: {result.optimal_point}")
print(f"Total evaluations : {len(energy_history)}")

## 7A.8 Visualization — VQE convergence

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(range(1, len(energy_history) + 1), energy_history,
        marker='o', markersize=4, linewidth=1.8,
        color='#58a6ff', label='VQE (UCCSD / STO-3G)')

ax.axhline(FCI_REF, linestyle='--', linewidth=1.5,
           color='#f78166', label=f'FCI ref = {FCI_REF} Ha')
ax.axhline(HF_REF,  linestyle=':', linewidth=1.5,
           color='#a5d6ff', label=f'HF  ref = {HF_REF} Ha')

ax.set_xlabel('Optimizer evaluation', fontsize=12)
ax.set_ylabel('Energy (Ha)', fontsize=12)
ax.set_title('VQE convergence for H₂ (STO-3G, R = 0.7414 Å)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_facecolor('#161b22')
fig.patch.set_facecolor('#0d1117')
ax.tick_params(colors='#8b949e')
ax.xaxis.label.set_color('#8b949e')
ax.yaxis.label.set_color('#8b949e')
ax.title.set_color('#e6edf3')

plt.tight_layout()
plt.savefig('vqe_convergence_h2_notebook.png', dpi=150, facecolor='#0d1117')
plt.show()
print("✓ Plot saved.")

## 7A.9 H₂ dissociation curve

We calculate the total energy for different internuclear distances $R$ and compare HF, exact FCI (PySCF), and VQE-UCCSD. The FCI − HF difference is the **correlation energy**, which UCCSD captures completely for 2-electron systems.

> **Note**: this calculation may take several minutes depending on the number of points and the machine.

In [ ]:
import time

def run_h2_vqe(bond_length: float, basis: str = 'sto3g') -> dict:
    """Calculates HF, FCI, and VQE-UCCSD for H₂ at a given distance R.

    Parameters
    ----------
    bond_length : float
        Internuclear distance in Å.
    basis : str
        Gaussian basis set (sto3g, 6-31g, cc-pvdz, ...).

    Returns
    -------
    dict with fields R, hf_total, fci_total, vqe_total, error_mHa.
    """
    geom   = f"H 0 0 0; H 0 0 {bond_length}"
    driver = PySCFDriver(atom=geom, basis=basis, charge=0, spin=0,
                         unit=DistanceUnit.ANGSTROM)
    res    = driver.run()

    n_part  = res.num_particles
    n_orb   = res.num_spatial_orbitals
    e_nuc   = float(res.nuclear_repulsion_energy)
    hf_elec = float(res.reference_energy)
    hf_tot  = hf_elec + e_nuc

    # Exact FCI using PySCF directly
    from pyscf import gto, scf, fci as pyscf_fci
    mol = gto.M(atom=geom, basis=basis, unit='Angstrom', verbose=0)
    mf  = scf.RHF(mol).run()
    cisolver = pyscf_fci.FCI(mf)
    fci_e, _ = cisolver.kernel()
    fci_tot  = float(fci_e)

    # VQE-UCCSD
    mapper  = JordanWignerMapper()
    q_ham   = mapper.map(res.hamiltonian.second_q_op())
    hf_st   = HartreeFock(n_orb, n_part, mapper)
    ans     = UCCSD(n_orb, n_part, mapper, initial_state=hf_st)
    vqe_run = VQE(StatevectorEstimator(), ans, SLSQP(maxiter=150, eps=1e-6))
    vqe_res = vqe_run.compute_minimum_eigenvalue(q_ham)
    vqe_tot = float(np.real(vqe_res.eigenvalue)) + e_nuc

    return dict(
        R=bond_length,
        hf_total=hf_tot,
        fci_total=fci_tot,
        vqe_total=vqe_tot,
        error_mHa=abs(vqe_tot - fci_tot) * 1000,
    )


# Curve points: 0.35 Å → 2.50 Å
R_values = np.linspace(0.35, 2.50, 18)
results  = []

for R in R_values:
    t0 = time.time()
    data = run_h2_vqe(R, basis='sto3g')
    elapsed = time.time() - t0
    results.append(data)
    print(f"R={R:.3f} Å  HF={data['hf_total']:.6f}  "
          f"FCI={data['fci_total']:.6f}  "
          f"VQE={data['vqe_total']:.6f}  "
          f"err={data['error_mHa']:.3f} mHa  ({elapsed:.1f}s)")

print("\n✓ Dissociation curve completed.")

In [ ]:
# ── Dissociation curve plot ────────────────────────────────────────
R_arr   = np.array([d['R']         for d in results])
hf_arr  = np.array([d['hf_total']  for d in results])
fci_arr = np.array([d['fci_total'] for d in results])
vqe_arr = np.array([d['vqe_total'] for d in results])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0d1117')

# ── Left panel: absolute energies ────
ax1.plot(R_arr, hf_arr,  '--', color='#a5d6ff', linewidth=1.8, label='HF')
ax1.plot(R_arr, fci_arr, '-',  color='#f78166', linewidth=2.0, label='Exact FCI')
ax1.plot(R_arr, vqe_arr, 'o', color='#58a6ff',  linewidth=1.8,
         markersize=5, linestyle='-', label='VQE-UCCSD')

ax1.axvline(0.7414, color='#888', linestyle=':', alpha=0.7,
            label='$R_{eq}$ = 0.7414 Å')
ax1.set_xlabel('Internuclear distance R (Å)', fontsize=11)
ax1.set_ylabel('Total energy (Ha)', fontsize=11)
ax1.set_title('H₂ dissociation curve (STO-3G)', fontsize=12)
ax1.legend(fontsize=9)
ax1.grid(alpha=0.25)
ax1.set_facecolor('#161b22')
ax1.tick_params(colors='#8b949e')

# ── Right panel: VQE vs FCI residual error ──────
err_mHa = (vqe_arr - fci_arr) * 1000
ax2.plot(R_arr, np.abs(err_mHa), 'o-', color='#e3b341',
         linewidth=1.8, markersize=5)
ax2.axhline(1.6, linestyle='--', color='#f78166', linewidth=1.2,
            label='Chemical accuracy (1.6 mHa)')
ax2.set_xlabel('Internuclear distance R (Å)', fontsize=11)
ax2.set_ylabel('|VQE vs FCI residual error| (mHa)', fontsize=11)
ax2.set_title('Residual correlation error', fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.25)
ax2.set_facecolor('#161b22')
ax2.tick_params(colors='#8b949e')

for ax in (ax1, ax2):
    ax.xaxis.label.set_color('#8b949e')
    ax.yaxis.label.set_color('#8b949e')
    ax.title.set_color('#e6edf3')

plt.tight_layout()
plt.savefig('vqe_dissociation_h2_notebook.png', dpi=150, facecolor='#0d1117')
plt.show()
print("✓ Plot saved.")

## 7A.10 Discussion of results

**Equilibrium geometry:**  
UCCSD is exact for 2 electrons, so the VQE converges to FCI with error $<$ 1 mHa (within **chemical accuracy**, the standard criterion of 1.6 mHa ≈ 1 kcal/mol).

**Dissociation curve:**  
At long distances ($R > 1.5$ Å) the multiconfigurational character of the state increases. UCCSD may occasionally fail if the SLSQP optimizer gets trapped in a local minimum, yielding larger errors.

**Resource scaling:**  

| Basis | Spatial orbitals | Qubits (JW) | UCCSD parameters |
|-------|-----------------|-------------|------------------|
| STO-3G | 2 | 4 | 3 |
| 6-31G | 4 | 8 | 15 |
| cc-pVDZ | 10 | 20 | ∼150 |

---

## 7A.11 Proposed exercises

1. **6-31G basis**: repeat the VQE calculation with the `6-31g` basis (8 qubits, 15 parameters). Does the result improve at equilibrium geometry? How does the runtime scale?

2. **COBYLA optimizer**: replace `SLSQP` with `COBYLA(maxiter=500)`. Compare the convergence and number of evaluations.

3. **Alternative ansatz**: use `RealAmplitudes(num_qubits=4, reps=2)` instead of UCCSD. Does it reach chemical accuracy? Why or why not?

4. **Noisy simulation**: repeat the VQE using `AerSimulator` with the noise model of a real IBM backend. Analyze the impact on the obtained energy.

5. **Correlation energy**: define the correlation energy as $E_c = E_{\text{FCI}} - E_{\text{HF}}$. What fraction does UCCSD capture at different distances?